In [22]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GEMINI_API_KEY    = os.getenv("GEMINI_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Gemini key loaded:   ", "✅" if GEMINI_API_KEY    else "❌ Missing!")

PageIndex key loaded: ✅
Gemini key loaded:    ✅


In [23]:
from pageindex import PageIndexClient
from google import genai

pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ PageIndex client ready")
print("✅ Gemini client ready")

✅ PageIndex client ready
✅ Gemini client ready


In [24]:
PDF_PATH = "./sample_content.pdf"

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: ./sample_content.pdf
✅ Uploaded!
📋 Document ID: pi-cmpmgcm9o03kw01qy4ehadfs4
   (Save this ID — you'll use it throughout the notebook)


In [25]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!


In [26]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Attention Is All You Need",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# Attention Is All You Need\nA 50-page educational overview inspired by the 2017 Transformer paper.\n\nThis document summarizes concepts related to the Transformer architecture, self-attention, training dynamics, applications, and historical impact.\n",
  "text": "# Attention Is All You Need\nA 50-page educational overview inspired by the 2017 Transformer paper.\n\nThis document summarizes concepts related to the Transformer architecture, self-attention, training dynamics, applications, and historical impact.\n",
  "nodes": [
    {
      "title": "Foundations and Mechanics of the Transformer Architecture",
      "node_id": "0001",
      "page_index": 1,
      "summary": "This text provides an overview of the Transformer architecture, detailing its departure from recurrent neural networks in favor of self-attention mechanisms. It explai

In [27]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Attention Is All You Need  (p.1)
  └─ [0001] Foundations and Mechanics of the Transformer Architecture  (p.1)
  └─ [0002] Core Components and Impact of the Transformer Architecture  (p.5)
  └─ [0003] Core Components and Architectural Impact of the Transformer  (p.9)
  └─ [0004] The Evolution and Impact of Transformer Architectures  (p.13)
  └─ [0005] The Evolution and Impact of Transformer Architectures  (p.17)
  └─ [0006] Transformer Architecture: Evolution, Applications, and Challenges  (p.21)
  └─ [0007] Evolution and Impact of the Transformer Architecture  (p.25)
  └─ [0008] Foundations and Evolution of the Transformer Architecture  (p.29)
  └─ [0009] Core Components and Impact of the Transformer Architecture  (p.33)
  └─ [0010] Core Components and Architectural Impact of the Transformer  (p.37)
  └─ [0011] The Transformer Architecture: Evolution, Applications, and Impact  (p.41)
  └─ [0012] The Evolution and Impact of Transformer Architectures  (

In [28]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 14
   Each node = one retrievable section of the document


In [29]:
from google import genai

genai_client = genai.Client(api_key=GEMINI_API_KEY)

In [32]:
import json

def llm_tree_search(query: str, tree: list, model: str = "gemini-2.5-flash") -> dict:
    """
    Core PageIndex retrieval:
    Sends query + document tree to Gemini.
    Returns:
        {
          "thinking": "...",
          "node_list": [...]
        }
    """

    def compress(nodes):
        out = []

        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title": n["title"],
                "page": n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }

            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])

            out.append(entry)

        return out


    compressed_tree = compress(tree)

    prompt = f"""
You are given a query and a document tree.

Identify which node IDs most likely contain the answer.

Reply ONLY as valid JSON:

{{
    "thinking": "<reasoning>",
    "node_list": ["node1","node2"]
}}

Query:
{query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}
"""


    response = genai_client.models.generate_content(
        model=model,
        contents=prompt
    )


    try:
        return json.loads(response.text)

    except Exception as e:
        print("Gemini raw output:")
        print(response.text)

        return {
            "thinking": f"Parse failed: {e}",
            "node_list": [],
            "raw_output": response.text
    }

In [33]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What is self attention mechanism ?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is self attention mechanism ?

Gemini raw output:
```json
{
    "thinking": "The query asks about the 'self attention mechanism'. I will look for nodes that explicitly mention 'Self-attention' in their summaries or titles. Nodes '0002' and '0009' both start their summaries with 'Page X: Self-attention', indicating they are highly relevant to the query.",
    "node_list": [
        "0002",
        "0009"
    ]
}
```
🧠 LLM Reasoning:
Parse failed: Expecting value: line 1 column 1 (char 0)

🎯 Selected Node IDs: []


In [34]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [35]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "gemini-2.5-flash") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs Gemini to cite section titles and page numbers.
    """

    if not nodes:
        return "⚠️ No relevant sections found in the document."

    # Build context string from retrieved nodes
    context_parts = []

    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )

    context = "\n\n---\n\n".join(context_parts)


    prompt = f"""
You are an expert document analyst.

Answer ONLY using the provided context.

Rules:
1. Cite section titles and page numbers for every claim.
2. If the answer isn't in the context, say:
   "The document does not contain sufficient information."
3. Be concise and precise.

Question:
{query}


Context:
{context}


Answer:
"""


    response = genai_client.models.generate_content(
        model=model,
        contents=prompt
    )


    return response.text.strip()

In [36]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [39]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What is self attention ?",
    tree=pageindex_tree
)

🔍 Query: What is self attention ?
Gemini raw output:
```json
{
    "thinking": "The query asks 'What is self attention?'. I need to find nodes that specifically mention 'self-attention'. Nodes 0002 and 0009 both have 'Self-attention' explicitly stated in their summaries, indicating they are highly relevant to answering the query.",
    "node_list": [
        "0002",
        "0009"
    ]
}
```

🧠 Reasoning: Parse failed: Expecting value: line 1 column 1 (char 0)...
🎯 Retrieved node IDs: []
📄 Sections found: []

📝 Answer:
⚠️ No relevant sections found in the document.
